# Paper Classification — Phase 1b M1 (TAPT) + Phase 1c Gate 2 verify

Pipeline:
1. Verify GPU (T4) + **disk ≥ 20GB free (Guard #4)**
2. Clone repo + install deps
3. Mount Drive + restore cached artifacts (data, .env, sanitize parquets, Phase A predictions, kNN parquets, **TAPT model**)
4. Phase 0 — sanitize (skip if cached)
5. Phase A — GPT-5 panel (skip if cached on Drive)
6. **M1 TAPT — check Drive first (Guard #3), only run if missing on Drive AND local**
7. Retrain 3 tasks (train→val) with TAPT-adapted encoder
8. Phase F — refit on train+val
9. **Phase 1c Gate 2 — evaluate (M1 + M2 cumulative)**
10. Print phase summary + drift comparison + download artifacts

**Time on T4 free tier**: ~8–10h. Within 12h session. All artifacts saved to Drive.

**Verification gate (Gate 2)**: After cell 9, test_2024 supp_macro_F1 (n≥10) must improve ≥ +0.02 vs `outputs/eval_report_baseline.json`. If not, rollback via `USE_TAPT=False` flag.

## 1. Verify GPU + disk (Guard #4)

In [ ]:
!nvidia-smi

In [ ]:
# Guard #4 — disk verify. TAPT writes a fresh SPECTER2 copy (~440MB) plus
# 3 task models × 1.3GB ≈ 4GB total. Free tier usually has 100+ GB free;
# require at least 20GB for safety.
import subprocess
out = subprocess.check_output(['df', '-BG', '/content']).decode()
print(out)
avail = int(out.splitlines()[1].split()[3].rstrip('G'))
assert avail >= 20, f'Need at least 20GB free, got {avail}GB. Restart runtime.'
print(f'Disk OK: {avail}GB free')

## 2. Clone repo + install dependencies

In [ ]:
import os
if not os.path.exists('paper-classification'):
    !git clone https://github.com/harnetlinh/paper-classification.git
%cd paper-classification
!git pull
!git log --oneline -5

In [ ]:
!pip install -q pyarrow openpyxl python-dotenv tenacity openai==1.57.4 transformers==4.45.2

## 3. Mount Drive + restore cached artifacts (Guard #3)

Drive path layout:
```
/content/drive/MyDrive/KHGDVN/
├── data/
│   └── 2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx
├── .env
└── outputs/
    ├── gold_2013_2023.parquet, main_2024_clean.parquet (Phase 0)
    ├── llm_classify/gpt5_panel_{val,test}.parquet (Phase A)
    ├── knn_retrieval/*.parquet (Phase D)
    └── specter2_tapt/ (Phase C / M1 — TAPT-adapted encoder)
```

Symlinking `outputs/` to Drive means every save touches Drive automatically. TAPT cell below checks `outputs/specter2_tapt/config.json` (which now points at Drive) and skips the 30-min training step if cached.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/KHGDVN'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)

import shutil
# One-time: move any locally produced outputs into Drive then symlink.
if os.path.exists('outputs') and not os.path.islink('outputs'):
    for item in os.listdir('outputs'):
        src = f'outputs/{item}'
        dst = f'{DRIVE_DIR}/outputs/{item}'
        if not os.path.exists(dst):
            shutil.move(src, dst)
    shutil.rmtree('outputs', ignore_errors=True)
if not os.path.islink('outputs'):
    os.symlink(f'{DRIVE_DIR}/outputs', 'outputs')

# Same pattern for data/ and .env
os.makedirs('data', exist_ok=True)
data_xlsx = '2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx'
drive_data = f'{DRIVE_DIR}/data/{data_xlsx}'
if os.path.exists(drive_data) and not os.path.exists(f'data/{data_xlsx}'):
    os.symlink(drive_data, f'data/{data_xlsx}')
if os.path.exists(f'{DRIVE_DIR}/.env') and not os.path.exists('.env'):
    shutil.copy(f'{DRIVE_DIR}/.env', '.env')

!ls -la outputs/ 2>/dev/null | head -20
!ls -la data/ 2>/dev/null

In [ ]:
# First-time setup: upload Excel + .env if not yet on Drive.
from google.colab import files
if not os.path.exists(f'data/{data_xlsx}'):
    print('Pick the Excel data file:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        os.rename(fn, f'data/{fn}')
        os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
        shutil.copy(f'data/{fn}', f'{DRIVE_DIR}/data/{fn}')

if not os.path.exists('.env'):
    print('Paste your .env content (OPENAI_API_KEY=sk-...). End with empty line + Ctrl+D:')
    env_content = ''
    try:
        while True:
            line = input()
            env_content += line + '\n'
    except EOFError:
        pass
    with open('.env', 'w') as f:
        f.write(env_content)
    shutil.copy('.env', f'{DRIVE_DIR}/.env')
print('Data + .env ready.')

## 4. Phase 0 — sanitize (skip if cached)

In [ ]:
if not os.path.exists('outputs/gold_2013_2023.parquet'):
    !python sanitize.py
else:
    print('Sanitize parquets already on Drive. Skipping.')
!ls outputs/*.parquet 2>/dev/null

## 5. Phase A — GPT-5 panel (skip if cached)

Cached predictions on Drive → re-runs are FREE.

In [ ]:
test_done = os.path.exists('outputs/llm_classify/gpt5_panel_test.parquet')
val_done = os.path.exists('outputs/llm_classify/gpt5_panel_val.parquet')
if not (test_done and val_done):
    !python llm_classify.py --split both
else:
    print('Phase A predictions cached. Skipping.')

## 6. M1 — TAPT (Drive checkpoint, Guard #3)

Skip if `outputs/specter2_tapt/config.json` exists (which now lives on Drive via the symlink). Re-run only when explicitly deleting the directory.

In [ ]:
# Guard #3: check Drive first via the symlinked outputs/specter2_tapt/.
# `config.json` is the most authoritative sentinel that save_pretrained()
# completed successfully (it's written last).
tapt_marker = 'outputs/specter2_tapt/config.json'
if os.path.exists(tapt_marker):
    print(f'TAPT artifact already on Drive: {tapt_marker} — skipping ~30min training.')
    !ls -la outputs/specter2_tapt/
else:
    print('TAPT artifact missing — running tapt.py (writes to Drive via symlink).')
    !python tapt.py --epochs 3 --lr 5e-5 --batch-size 16

## 7. Retrain 3 tasks (train→val) with TAPT-adapted encoder

`train_specter2.py:train_model()` auto-loads from `outputs/specter2_tapt/` when `USE_TAPT=True` and the path exists (default in `config.py`). Log line confirms.

**Note**: deleting `outputs/model_*.pt` from Drive between runs is the way to force a clean re-train with the TAPT encoder. If existing `model_{task}.pt` are HF-Hub-trained, evaluate.py would still load them — best practice is to delete first.

In [ ]:
# To force fresh train with TAPT encoder, delete old checkpoints:
# !rm -f outputs/model_*.pt outputs/thresholds_*.json
!python train_specter2.py --task all

## 8. Phase F — refit on train+val

In [ ]:
!python train_specter2.py --task all --include-val-in-train

## 9. Phase 1c Gate 2 — evaluate (M1 + M2 cumulative)

Output `outputs/eval_report.json`. Quantification (M2) automatically applied to test split when `USE_QUANTIFICATION_AT_TEST=True` (default). Phase A panel + 3-way ensemble auto-detect cached parquets.

In [ ]:
!python evaluate.py --task all

## 10. Phase summary + Gate 2 verification

Compare current report against `outputs/eval_report_baseline.json` (pre-M1, pre-M2). Gate threshold: test_2024 supp_macro_F1 (n≥10) must improve ≥ +0.02 for **Fields task**.

In [ ]:
!python phase_summary.py

In [ ]:
# Automated Gate 2 check — compare current vs baseline.
import json, numpy as np
with open('outputs/eval_report.json') as f:
    cur = json.load(f)
baseline_path = 'outputs/eval_report_baseline.json'
if not os.path.exists(baseline_path):
    print('No baseline to compare against. Skipping Gate 2 check.')
else:
    with open(baseline_path) as f:
        base = json.load(f)
    print(f'{"Task":<10} {"baseline_n>=10":>14} {"current_n>=10":>14} {"delta":>10} {"gate (>=+0.02)":>16}')
    print('-' * 65)
    def supp10(task_report):
        t = task_report.get('test_2024', {})
        sup = t.get('support', [])
        pcf = t.get('per_class_f1', [])
        idx = [i for i, s in enumerate(sup) if s >= 10]
        if not idx or not pcf: return float('nan')
        return float(np.mean([pcf[i] for i in idx]))
    overall_pass = True
    for task in ['fields', 'levels', 'method']:
        b = supp10(base.get('tasks', {}).get(task, {}))
        c = supp10(cur.get('tasks', {}).get(task, {}))
        d = c - b if not (np.isnan(b) or np.isnan(c)) else float('nan')
        gate = 'PASS' if d >= 0.02 else 'FAIL' if d < 0 else 'MARGIN'
        if task == 'fields' and gate != 'PASS':
            overall_pass = False
        print(f'{task:<10} {b:>14.4f} {c:>14.4f} {d:>+10.4f} {gate:>16}')
    print()
    if overall_pass:
        print('Gate 2 OVERALL: PASS — proceed to Phase 2 (A modified, mask source title)')
    else:
        print('Gate 2 OVERALL: FAIL on Fields. Consider rollback:')
        print('  1. USE_TAPT=False (revert encoder load)')
        print('  2. Re-run eval to isolate M2-only impact')
        print('  3. If still no gain, USE_QUANTIFICATION_AT_TEST=False')

## 11. Export human-review workbook

In [ ]:
!python export_review.py --output outputs/review.xlsx

## 12. Download artifacts (small files only — TAPT + models stay on Drive)

In [ ]:
!zip -r artifacts.zip outputs/eval_report.json outputs/training_log_*.json outputs/thresholds_*.json outputs/review.xlsx 2>/dev/null
from google.colab import files
files.download('artifacts.zip')

## Resume after disconnect

All heavy artifacts (TAPT encoder ~440MB, model checkpoints ~1.3GB × 3 tasks, Phase A predictions) live on Drive via the `outputs/` symlink. Re-running cells 1, 2, 3 restores the symlink and every subsequent cell skips its expensive step when its sentinel file exists.

Manual force-rerun:
- TAPT: `!rm -rf outputs/specter2_tapt/`
- Train: `!rm -f outputs/model_*.pt outputs/thresholds_*.json outputs/training_log_*.json`
- Phase F: `!rm -f outputs/model_*_trainval.pt`
- Phase A: `!rm -rf outputs/llm_classify/`